# RFSoC 4×2 雙板即時攝影機 · 4 子載波 16-QAM
A 板將 USB 攝影機的 640 × 480 灰階 JPEG 裝入 307,200-byte BRAM 區塊。影像走 A 的 DAC_A → B 的 ADC_A，回覆走 B 的 DAC_A → A 的 ADC_A。兩板使用同一份 overlay。使用同軸時，依板卡規格在兩條路徑各配置適當衰減。

1. 把同一份部署包解壓到兩板各自的 Jupyter 資料夾，從該目錄開啟 notebook。攝影機接 A 板。
2. B 板用預設的 `ROLE = "receive"` 執行所有儲存格。看到 `RX_WAITING_FOR_PILOT` 再於 A 板設定 `ROLE = "send"` 並執行。
3. 錄影時間由你決定。要正常結束，**在 A 板按 Jupyter 的 ■「Interrupt Kernel」**。A 會收尾並送出結束封包，B 會自動完成 MP4。B 也可單獨中斷並保留已寫入的畫面。如兩板分別中斷，下次串流請依上述順序重新執行。

In [ ]:
from pathlib import Path
from webcam_live import run_live

ROLE = "receive"                 # B: "receive"；A: "send"
BITFILE = Path("ofdm_qam16.bit")
CAMERA = 0                       # 僅 A 使用。也可填影片路徑，播完即結束
FPS = 30.0                       # A 的擷取目標幀率，B 會讀取 A 送來的設定
AMPLITUDE = 4096
LIVE_PREVIEW = True
OUTPUT_ROOT = Path("received_live_webcam")

### 啟動收發
A 和 B 各自量測對方的校準訊號。預覽在 A 顯示將送出的 JPEG，在 B 顯示實際收到並通過完整性檢查的畫面。當相機快於鏈路，A 最多暫存 22 張 JPEG，優先保留近期畫面。B 的 MP4 按 A 的擷取時間播放，在掉幀的間隔延用上一張已收到的畫面。

In [ ]:
summary = run_live(ROLE, BITFILE, OUTPUT_ROOT, camera=CAMERA, fps=FPS,
                   amplitude=AMPLITUDE, show_preview=LIVE_PREVIEW)
summary

每次執行都有獨立結果目錄。B 的 `live_received.mp4` 與 `summary.json` 位於 `received_live_webcam/<時間>_receive/`，摘要包含接收張數、捨棄區塊數和是否收到結束封包。A 端摘要記錄送出張數和佇列丟幀數。

若要用手邊的雙板 32 子載波版本做對照，可將原本的 `ofdm_air8_rfsoc4x2.zip` 解到子目錄，兩板的 `BITFILE` 同時改指向該處的 `ofdm_air8.bit`。程式會從該包的 manifest 核對硬體版本，並顯示其 QPSK/BPSK 設定。原本單板 Method B 的 `ofdm_rf.bit` 是本版的攝影機流程參考。